# Matrix completion via recommendation system example

This example demonstrates the use of matrix completion techniques on a recommendation system.  The recommendation system uses data from the [360K Last.fm dataset](http://ocelma.net/MusicRecommendationDataset/lastfm-360K.html).

In [2]:
!pip install -U implicit h5py

   ---------------------------------------- 0.0/748.6 kB ? eta -:--:--
   ---------------------------------------- 748.6/748.6 kB 10.4 MB/s  0:00:00
   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   ---------------------------------------- 3.2/3.2 MB 20.9 MB/s  0:00:00

   ---------------------------------------- 0/3 [tqdm]
   ------------- -------------------------- 1/3 [h5py]
   ------------- -------------------------- 1/3 [h5py]
   ------------- -------------------------- 1/3 [h5py]
   ------------- -------------------------- 1/3 [h5py]
   ------------- -------------------------- 1/3 [h5py]
   -------------------------- ------------- 2/3 [implicit]
   -------------------------- ------------- 2/3 [implicit]
   ---------------------------------------- 3/3 [implicit]



In [3]:
# retrieving last.fm dataset
from implicit.datasets.lastfm import get_lastfm
import numpy as np
import pandas as pd
from scipy import sparse
import os
from pathlib import Path

c:\Users\tziga\miniconda3\envs\de300\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Downloading and saving the Last.fm dataset

In [4]:
filepath = r'datasets/'
Path(filepath).mkdir(exist_ok=True)

if not os.path.exists(filepath + r'artist_user_plays.npz'):
    # save our dataset in sparse format
    artists, users, artist_user_plays = get_lastfm()

    sparse.save_npz(filepath + r'artist_user_plays.npz', artist_user_plays)
    np.save(filepath + 'artists.npy', artists)
    np.save(filepath + 'users.npy', users)
else:
    # load our dataset into original format
    artist_user_plays = sparse.load_npz(filepath + r'artist_user_plays.npz')
    artists = np.load(filepath + 'artists.npy', allow_pickle=True)
    users = np.load(filepath + 'users.npy', allow_pickle=True)

0.00B [00:00, ?B/s]

184MB [00:14, 12.9MB/s]                              


In [11]:
# investigate the content of the downloaded dataset
artists, users, artist_user_plays

(array([' 2 ', ' 58725ab=>', ' 80lİ yillarin tÜrkÇe sÖzlÜ aŞk Şarkilari',
        ..., '��|', '��疲暎�', '�������'], shape=(292385,), dtype=object),
 array(['00000c289a1829a808ac09c00daf10bc3c4e223b',
        '00001411dc427966b17297bf4d69e7e193135d89',
        '00004d2ac9316e22dc007ab2243d6fcb239e707d', ...,
        'ffff9af9ae04d263dae91cb838b1f3a6725f5ffb',
        'ffff9ef87a7d9494ada2f9ade4b9ff637c0759ac', 'sep 20, 2008'],
       shape=(358868,), dtype=object),
 <Compressed Sparse Row sparse matrix of dtype 'float32'
 	with 17535606 stored elements and shape (292385, 358868)>)

In [12]:
# return the dimensions of data
artists.shape[0], users.shape[0], artist_user_plays.shape

(292385, 358868, (292385, 358868))

In [16]:
# return the number of non-missing entries 
artist_user_plays.count_nonzero()

17535605

In [17]:
# investigate the proportion of non-zero entries
artist_user_plays.count_nonzero() / np.prod(artist_user_plays.shape) # prod takes product of array elements

np.float64(0.0001671209636692248)

## Preparing the data
Okapi BM25 (Best Matching) scoring is a ranking algorithm used by search engines to estimate the relevance of items to a given search query, based on the frequency of occurrences and the size of the reference pool.  The origin of the algorithm is used in search terms in a pool of documents.

For completeness, the BM25 score of query $Q=\{q_1, \ldots, q_n\}$ for a document $D$ is calculated as:

$$\text{BM25}(D, Q) = \sum_{i=1}^{n} \frac{IDF(q_i) \cdot f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot (1 - b + b \cdot \frac{|D|}{\text{avgD}})},$$
where
- $IDF(q_i)$ is the inverse document frequency of term $q_i$.
- $f(q_i, D)$ is the term frequency of $q_i$ in the document $D$.
- $k_1$ and $b$ are parameters controlling term saturation and document length normalization.
- $D$ is the length of the document.
- $\text{avgD}$ is the average document length in the corpus.


In [18]:
from implicit.nearest_neighbours import bm25_weight

# using the weighting function for normalization
artist_user_plays = bm25_weight(artist_user_plays, K1=100, B=0.8)
user_plays = artist_user_plays.T.tocsr()

## Training the model with alternating least squares

In [19]:
from implicit.als import AlternatingLeastSquares

# using alternating least squares algorithm
model = AlternatingLeastSquares(factors = 16, regularization=0.05, alpha=2.0)
model.fit(user_plays)

c:\Users\tziga\miniconda3\envs\de300\lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 15/15 [00:38<00:00,  2.57s/it]


## Similar artists recommendation

In [22]:
# generate similar artist recommendation
artist_id = list(artists).index('beyonce')
ids, scores = model.similar_items(artist_id)
pd.DataFrame({'artist': artists[ids], 'score': scores})


,artist,score
0,beyonce,1.000000
1,bayje,0.989434
2,ina,0.979235
3,dj l-gee,0.975745
4,m. pokora,0.975379
5,ivena,0.975089
6,akon feat. colby o donis and kardinal offishall,0.975020
7,charlie wilson & pharrell,0.974716
8,esmée denters,0.969594
9,j-status ft rihanna and shontelle,0.968845


## User-specific recommendation

In [23]:
# generate user-based recommendation
userid = 10
ids, scores = model.recommend(userid, user_plays[userid], N=10)
artists[ids]

array(['anna ternheim', 'hello saferide', 'moneybrother', 'miss li',
       'marit bergman', 'veronica maggio', 'broder daniel', 'laakso',
       'familjen', 'maia hirasawa'], dtype=object)